In [1]:
import numpy as np
import pandas as pd
import re
import pickle
import json
import time
import os
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Layer, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
tf.random.set_seed(42)
np.random.seed(42)

In [3]:
df = pd.read_csv("Dataset_English_Hindi.csv")
df.drop_duplicates(inplace=True)
df = df.dropna()
df["English"] = df["English"].astype(str)
df["Hindi"]   = df["Hindi"].astype(str)
print(f"Total pairs: {len(df)}")

Total pairs: 127375


In [4]:
def clean_english(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[^a-zA-Z0-9\s]', '', sentence)
    sentence = re.sub(r'https?://\S+|www\.\S+', '', sentence)
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    sentence = emoji_pattern.sub(r'', sentence)
    return sentence.strip()

In [5]:
def clean_hindi(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r'[^\u0900-\u097F\s]', '', sentence)
    sentence = re.sub(r'https?://\S+|www\.\S+', '', sentence)
    return sentence.strip()

In [6]:
df["English"] = df["English"].apply(clean_english)
df["Hindi"]   = df["Hindi"].apply(clean_hindi)

In [7]:
# Filter by sentence length for efficient training
MAX_ENG_WORDS = 25
MAX_HIN_WORDS = 30
mask = (
    df["English"].str.split().str.len().between(5, MAX_ENG_WORDS) &
    df["Hindi"].str.split().str.len().between(5, MAX_HIN_WORDS)
)
df = df[mask].reset_index(drop=True)
print(f"Pairs after length filter: {len(df)}")

Pairs after length filter: 88576


In [8]:
# Add <start> / <end> tokens to Hindi
df["Hindi_in"]  = "<sos> " + df["Hindi"]          # decoder input
df["Hindi_out"] = df["Hindi"] + " <eos>"          # decoder target

In [9]:
# English tokenizer
eng_tok = Tokenizer(filters="", lower=True, oov_token="<OOV>")
eng_tok.fit_on_texts(df["English"])
ENG_VOCAB = len(eng_tok.word_index) + 1

In [10]:
# Hindi tokenizer  (case-sensitive for Devanagari)
hin_tok = Tokenizer(filters="", lower=False, oov_token="<OOV>")
hin_tok.fit_on_texts(df["Hindi_in"].tolist() + df["Hindi_out"].tolist())
HIN_VOCAB = len(hin_tok.word_index) + 1

In [11]:
print(f"English vocab: {ENG_VOCAB}  |  Hindi vocab: {HIN_VOCAB}")

English vocab: 52611  |  Hindi vocab: 54090


In [12]:
df_enc_inp = pad_sequences(
    eng_tok.texts_to_sequences(df["English"]),
    maxlen = MAX_ENG_WORDS, padding="post"
)
df_dec_inp = pad_sequences(
    hin_tok.texts_to_sequences(df["Hindi_in"]),
    maxlen = MAX_HIN_WORDS + 1, padding="post"
)
df_dec_out = pad_sequences(
    hin_tok.texts_to_sequences(df["Hindi_out"]),
    maxlen = MAX_HIN_WORDS + 1, padding="post"
)

In [13]:
ENG_SEQ_LEN = df_enc_inp.shape[1]
HIN_SEQ_LEN = df_dec_inp.shape[1]
print(f"Encoder seq len: {ENG_SEQ_LEN}  |  Decoder seq len: {HIN_SEQ_LEN}")
print(f"df_enc_inp shape: {df_enc_inp.shape}")
print(f"df_dec_out shape: {df_dec_out.shape}")

Encoder seq len: 25  |  Decoder seq len: 31
df_enc_inp shape: (88576, 25)
df_dec_out shape: (88576, 31)


In [14]:
#Model Hyper Parameters
EMBED_DIM = 512
BATCH_SIZE = 32
EPOCHS = 50

In [15]:
#Encoder
enc_input = Input(shape=(ENG_SEQ_LEN,))
enc_embed = Embedding(ENG_VOCAB, EMBED_DIM)(enc_input)
enc_lstm = LSTM(EMBED_DIM, return_state=True)
enc_out, enc_h, enc_c = enc_lstm(enc_embed)
enc_model = Model(enc_input, [enc_out, enc_h, enc_c])

In [16]:
#Decoder
dec_input = Input(shape=(HIN_SEQ_LEN,))
dec_embed  = Embedding(HIN_VOCAB, EMBED_DIM, name="dec_embedding")(dec_input)
dec_lstm = LSTM(EMBED_DIM, return_sequences=True, return_state=True)
dec_out, dec_h, dec_c = dec_lstm(dec_embed, initial_state=[enc_h, enc_c])
dec_dense  = Dense(HIN_VOCAB, activation='softmax', name="dec_dense")(dec_out)

In [17]:
#Model
model = Model([enc_input, dec_input], dec_dense)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 25)]         0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 31)]         0           []                               
                                                                                                  
 embedding (Embedding)          (None, 25, 512)      26936832    ['input_1[0][0]']                
                                                                                                  
 dec_embedding (Embedding)      (None, 31, 512)      27694080    ['input_2[0][0]']                
                                                                                            

In [18]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
early_stopping = EarlyStopping(monitor='accuracy', patience=5, restore_best_weights=True)

In [19]:
#Training
history = model.fit(
    x=[df_enc_inp, df_dec_inp],
    y=df_dec_out,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks = [early_stopping]
)

Epoch 1/50
 351/2768 [==>...........................] - ETA: 2:52 - loss: 3.7413 - accuracy: 0.5580


KeyboardInterrupt



In [ ]:
model.save("my_model.keras")
model.save_weights("my_model.weights.h5")
with open("my_model_architecture.json", 'w') as f:
    f.write(model.to_json())

history_df = pd.DataFrame(history.history)
history_df.index      = range(1, len(history_df) + 1)   # epochs start from 1
history_df.index.name = "epoch"
history_df.to_csv("training_history.csv")

In [ ]:
#Inference
encoder_inf_model = Model(enc_input, [enc_h, enc_c])
dec_emb_layer   = model.get_layer("dec_embedding")
dec_dense_layer = model.get_layer("dec_dense")

dec_inf_input   = Input(shape=(1,))
dec_state_inp_h = Input(shape=(EMBED_DIM,))
dec_state_inp_c = Input(shape=(EMBED_DIM,))

dec_inf_embed = dec_emb_layer(dec_inf_input)
dec_inf_out, dec_inf_h, dec_inf_c = dec_lstm(dec_inf_embed, initial_state=[dec_state_inp_h, dec_state_inp_c])
dec_inf_dense = dec_dense_layer(dec_inf_out)

decoder_inf_model = Model(
    inputs  = [dec_inf_input, dec_state_inp_h, dec_state_inp_c],
    outputs = [dec_inf_dense, dec_inf_h, dec_inf_c]
)

In [ ]:
#Translate
hin_idx2word = {idx: word for word, idx in hin_tok.word_index.items()}
sos_token    = hin_tok.word_index["<sos>"]
eos_token    = hin_tok.word_index["<eos>"]

In [ ]:
def translate(english_sentence, max_len=MAX_HIN_WORDS):
    # Step 1: Clean and pad English input
    eng_clean = clean_english(english_sentence)
    eng_seq   = eng_tok.texts_to_sequences([eng_clean])
    eng_pad   = pad_sequences(eng_seq, maxlen=ENG_SEQ_LEN, padding="post")

    # Step 2: Encode → get initial states
    h, c = encoder_inf_model.predict(eng_pad, verbose=0)

    # Step 3: Start decoding with <sos>
    target_token    = np.zeros((1, 1))
    target_token[0, 0] = sos_token

    translated = []

    # Step 4: Decode one token at a time
    for _ in range(max_len):
        pred, h, c = decoder_inf_model.predict(
            [target_token, h, c], verbose=0
        )

        predicted_id   = np.argmax(pred[0, 0, :])
        predicted_word = hin_idx2word.get(predicted_id, "")

        if predicted_id == eos_token or predicted_word == "":
            break

        translated.append(predicted_word)
        target_token[0, 0] = predicted_id   # feed output as next input

    return " ".join(translated)

In [ ]:
#Test
test_sentences = [
    "How are you doing today",
    "I want to eat food",
    "The weather is very cold outside",
    "She is going to the market",
    "My name is Rahul",
    "I love my country",
    "He is a very good student",
    "Please open the door",
    "I am feeling very tired today",
    "The dog is running in the park",
    "Can you help me please",
    "I want to drink water",
    "She loves to read books",
    "The sun rises in the east",
    "I go to school every day",
    "He does not like cold weather",
    "We are going to the temple",
    "The train is very late today",
    "I want to sleep now",
    "My mother cooks very delicious food"
]

for s in test_sentences:
    print(f"English : {s}")
    print(f"Hindi   : {translate(s)}")
    print("-" * 40)